In [25]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

from openai import OpenAI
openai_client = OpenAI()

In [26]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [27]:
from rag_helper import RAGBase

instructions = """
You are a helpful course assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions
)

In [28]:
answer1 = assistant.rag('How do i run Ollama locally?')
print(answer1)

To run Ollama locally:

1. **Install Ollama**
   - Go to https://ollama.com/download
   - Choose your OS:
     - **macOS**: download the `.pkg` and install it
     - **Windows**: download the `.msi` and install it
     - **Linux**: run:
       ```bash
       curl -fsSL https://ollama.com/install.sh | sh
       ```

2. **Start a model locally**
   Open a terminal and run:
   ```bash
   ollama run llama3
   ```
   This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

3. **Test the local server**
   Run:
   ```bash
   curl http://localhost:11434
   ```
   You should get a response like:
   ```json
   {"models": [...]}  
   ```

If you’re using it from Python, you can also install the client with:

```bash
pip install ollama
```


In [29]:
answer2 = assistant.rag("How do I run Olama locally?")
print(answer2)

I couldn’t find anything about “Olama” in the FAQ context.

If you meant running the course locally, the FAQ says you can do that if you’re comfortable setting up Python, `uv`, Jupyter, Docker, and any other tools needed for the module. If you run locally, make sure you document your setup and keep your environment reproducible.


In [30]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course, can I join?'},
]

response = openai_client.responses.create(
    model = 'gpt-5.4-mini',
    input = messages
)

print(response.output_text)

Yes, probably — but it depends on the course’s enrollment rules and whether there’s still space.

If you want, I can help you figure out the best way to ask. A simple message could be:

> Hi, I just discovered this course and I’m very interested in joining. Is it still possible to enroll at this stage?

If you’re asking me to contact someone or draft a more specific email/message, tell me:
- the course name
- whether it’s for school, work, or an online program
- how formal you want the message to sound

I can write it for you.


In [31]:
def search(query):
    boost_dict = {'question':3.0, 'section':0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

search('How do I run Olama locally?')

[{'id': 'aa310de435',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Can I run the course locally instead of Codespaces?',
  'answer': 'Yes. Codespaces is just the easiest way for everyone to start with the same environment.\n\nYou can run the course locally if you are comfortable setting up Python, `uv`, Jupyter, Docker, and any other tools needed for the module.\n\nIf you run locally, make sure you document your setup and keep your environment reproducible.'},
 {'id': '193612db63',
  'course': 'llm-zoomcamp',
  'section': 'Module 3: Orchestration',
  'question': "Why do we need orchestration / Kestra — can't I just run the code in a notebook?",
  'answer': "Notebooks are great for learning and experimenting, but real AI workflows need more than a script that runs once: scheduling, retries, monitoring, secret management, and reliably chaining tasks together. That's what an orchestrator like Kestra provides.\n\nIn this module Kestra is also the vehicle for the

In [32]:
# Now we tell the LLM about this search function_tool (via python dictionary) we created above. We only provide a schema describing what the function does and what arguments it takes. T
# The LLM will then be able to call this function when needed. 

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for relevant information regarding the given query.",
    'parameters':{
        "type": "object",
        "properties": {
            "query" : {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}


response = openai_client.responses.create(
    model = 'gpt-5.4-mini',
    input = messages,
    tools = [search_tool]
)

len(response.output)

1

In [33]:
call = response.output[0]
print(call)

ResponseFunctionToolCall(arguments='{"query":"discovered course can I join enrollment join late registration join the course"}', call_id='call_hGH2qJR9c3YyEV2G2YiaHYV3', name='search', type='function_call', id='fc_026dbb64a75ea66b006a74cf2ad16081a2b7f7c08e061e1c42', namespace=None, status='completed')


In [34]:
call.name

'search'

In [35]:
import json
args = json.loads(call.arguments)
args

{'query': 'discovered course can I join enrollment join late registration join the course'}

In [36]:
# This will be sent to LLM 
result = search(**args)
result_json = json.dumps(result, indent=2)
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "977bf7786c",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?",
    "answer": "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."
  },
  {
    "id": "04919992b3",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "How should I start the course and follow the weekly 

In [37]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

In [38]:
messages.append(call)
messages.append(function_call_output)
messages

[{'role': 'user', 'content': 'I just discovered the course, can I join?'},
 ResponseFunctionToolCall(arguments='{"query":"discovered course can I join enrollment join late registration join the course"}', call_id='call_hGH2qJR9c3YyEV2G2YiaHYV3', name='search', type='function_call', id='fc_026dbb64a75ea66b006a74cf2ad16081a2b7f7c08e061e1c42', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_hGH2qJR9c3YyEV2G2YiaHYV3',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "977bf7786c",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Course: I have registered for the LLM Zoomcamp. When can I expec

In [39]:
response = openai_client.responses.create(
    model = 'gpt-5.4-mini',
    input = messages,
    tools = [search_tool]
)

print(response.output_text)

Yes, you can still join and start learning anytime.

If you want a certificate, though, you need to submit your project while the course is still accepting submissions.


In [40]:
response.usage.input_tokens, response.usage.output_tokens, response.usage.total_tokens


(775, 37, 812)

In [41]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    # Prices per 1M tokens (example pricing)
    INPUT_PRICE_PER_MILLION = 0.15   # $0.15 / 1M input tokens
    OUTPUT_PRICE_PER_MILLION = 0.60  # $0.60 / 1M output tokens

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }


# Your tokens
result = calculate_gpt54mini_price(777, 37)

print("Total Cost: $", round(result["total_cost"], 8))

Total Cost: $ 0.00013875


History: 
1. Make a call to LLM <-- pay
2. LLM decided to invoke search('params')
3. We invoke the search, we have the results
4. Send the results back to the LLM (2nd call) <--- pay
5. LLM processes the results
6. LLM gives the answer


2 requests sent to LLM

## Why we need Loops - Agentic Loops
--> After LLM processes the results, it decides to make another tool call / several calls
1. Make a call to LLM <-- pay
2. LLM decided to invoke search('params')
3. We invoke the search, we have the results
4. Send the results back to the LLM (2nd call) <--- pay
5. LLM processes the results
6. LLM decides to make another tool call 
7. Send one more API request <----
8. LLM processes & gives the answer


--> By the way we dont know how many calls the llm will make, it wil loop untill they are no tool calls to make

In [42]:
# Function call helper - function to handle the function call from the LLM and return the output in a structured format.
# converts json string of arguments to python dictionary and calls the search function with the arguments.
#  It then returns the output in a structured format.

def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [43]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()


In [44]:

question = 'I just discovered the course. Can I join it?'

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course late enrollment discovered the course can I join FAQ"}


In [22]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool],
)

print(response.output_text)

Yes — you can still join the course.

If your goal is a certificate, the important thing is to submit your project while submissions are still being accepted. You can also start learning and working through the materials anytime.

If you want, I can also help with:
- how to start the course,
- whether you can get a certificate,
- or how the weekly workflow works.

Is there anything else you’d like to explore?


In [23]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course after course started enrollment FAQ discovered the course can I join"}', call_id='call_zhV3q9GuivrmIrtnFL7fGZFw', name='search', type='function_call', id='fc_0a70ba199c779978006a74c1541c38819fbc607687c9948041', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"can I join course late enrollment FAQ discover course after sta

## Full Loop

In [45]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=messages,
        tools=[search_tool]
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function_call:', item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == 'message':
            print('ASSISTANT:')
            print(item.content[0].text)
    
    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"join course discovered course can I join enrollment FAQ"}
function_call: search {"query":"late enrollment course join after start FAQ discovered course"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

You can start learning anytime, and the videos/materials are available. If you want a certificate, make sure you submit your project while submissions are still open.

If you’d like, I can also help you figure out how to start the course or explain the certificate requirements.


In [46]:
def agent_loop(instructions, question, model='gpt-5.4-mini') -> str:
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                print('ASSISTANT:')
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break
    
    return last_answer

In [47]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [48]:
result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"join the course enrollment late discovered course can I join"}
iteration #2...
function_call: search {"query":"certificate project submission while accepting submissions self-paced join course accepted no need register live cohort"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

A couple of important notes:
- You can start learning even if you discovered it late.
- If you want a certificate, you need to submit your project while submissions are still being accepted.

If you want, I can also help you figure out the best way to start from here or explain the certificate requirements.


In [49]:
result

'Yes — you can still join the course.\n\nA couple of important notes:\n- You can start learning even if you discovered it late.\n- If you want a certificate, you need to submit your project while submissions are still being accepted.\n\nIf you want, I can also help you figure out the best way to start from here or explain the certificate requirements.'

In [50]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "what's queen gambit?"

result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"queen gambit chess opening queen's gambit"}
iteration #2...
function_call: search {"query":"queen gambit queen's gambit course FAQ"}
iteration #3...
ASSISTANT:
I couldn’t find a course FAQ entry for “queen gambit,” so this looks off-topic for the course.

If you meant **Queen’s Gambit** in chess, it’s a chess opening. But I can only answer course/FAQ-based questions here.

Would you like to explore any course-related topics instead?


In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [ ]:
response = openai_client.responses.create(
    model = 'gpt-5.4-mini',
    input = messages,
    tools = [search_tool]
)

#response.output


In [ ]:


messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)

In [ ]:
# messages.extend(response.output)

# for item in response.output:
#     print(item)

#     if item.type == 'function_call':
#         print(f"Function call detected -->: {item.name}. Arguments detected--> : {item.arguments}")
#         call_output = make_call(item)
#         messages.append(call_output)
        
#     elif item.type == 'message':
#         print('ASSISTANT:')
#         print (item.content[0].text)
        

Yes — you can still join the course.

If your goal is just to learn, you can start anytime. If you want a certificate, you’ll need to submit your project while the course is still accepting submissions.

You can also begin using the course materials right away, even if you discovered it late.

If you want, I can also help you figure out the best way to start from where you are now.


In [ ]:
# messages

In [ ]:
# --> Processing one response <-- 